# Colab Model Station（模型工作站）

專為 Google Colab 與雲端 GPU 環境設計的自動化、可重現推論部署框架。
支援多種推論後端（**FastAPI + Diffusers** / **ComfyUI Headless API**），並具備動態掛載與切換 **FLUX**、**SDXL** 與 **LoRA** 的能力，可透過 **Tailscale Mesh 私網直連** 或 **Cloudflare 公網通道** 進行呼叫。

---

## 步驟 0：專案初始化與硬體檢測
將 `colab-inference-node` 複製至執行階段儲存空間（`/content/colab-inference-node`），注入 Colab Secrets 機密，並檢測 GPU 硬體配置。

In [ ]:
# ==============================================================================
# 步驟 0：複製儲存庫、注入機密與檢測硬體
# ==============================================================================
import os
from google.colab import userdata

# 1. 從 Colab Secrets 讀取機密（若已設定）
try:
    os.environ['TAILSCALE_AUTHKEY'] = userdata.get('TAILSCALE_AUTHKEY')
except Exception:
    pass

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. 複製或更新儲存庫
!git clone https://github.com/yixuanw99/colab-inference-node.git /content/colab-inference-node 2>/dev/null || (cd /content/colab-inference-node && git pull)

# 3. 輸出硬體狀態報告
!cd /content/colab-inference-node && bash engine.sh status


## 步驟 1：環境依賴安裝

依據目標工作流程安裝對應相依套件：
- **僅安裝 Diffusers** (`bash setup.sh diffusers`)：安裝 FastAPI、Uvicorn、Diffusers、Transformers、PEFT 與 Accelerate。
- **僅安裝 ComfyUI** (`bash setup.sh comfyui`)：複製 ComfyUI 核心並安裝執行相依套件。
- **完整安裝** (`bash setup.sh all`)：安裝 Diffusers、Tailscale、Cloudflared 與系統核心工具。

In [ ]:
%%bash
cd /content/colab-inference-node

# 選擇安裝模式：diffusers | comfyui | all
bash setup.sh all


## 硬體規格、資源佔用與推薦模型矩陣

請對照獲配的 GPU 等級與 VRAM 預算，選擇最佳精度與卸載策略：

| GPU 等級 | 顯存 (VRAM) | 推論引擎 | 推薦模型 | 精度格式 | 記憶體卸載模式 | 推論步數 | 預估耗時 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **NVIDIA T4** | 15 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | 序列循序卸載 (Sequential) | 4 | ~35秒 - 50秒 |
| **NVIDIA T4** | 15 GB | **Diffusers** | `stabilityai/stable-diffusion-xl-base-1.0` | FP16 | 模型卸載 (Model Offload) | 30 | ~12秒 - 18秒 |
| **NVIDIA L4** | 24 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | 模型卸載 (Model Offload) | 4 | ~12秒 - 18秒 |
| **NVIDIA L4** | 24 GB | **Diffusers** | `black-forest-labs/FLUX.1-dev` | FP8 | 模型卸載 (Model Offload) | 28 | ~45秒 - 65秒 |
| **NVIDIA A100** | 40/80 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | BF16 | 全顯存常駐 (None) | 4 | ~4秒 - 8秒 |
| **NVIDIA A100** | 40/80 GB | **Diffusers** | `black-forest-labs/FLUX.1-dev` | BF16 | 全顯存常駐 (None) | 28 | ~15秒 - 25秒 |

---

## 部署方案 A：Diffusers 服務 + Tailscale Mesh 私網直連（推薦）

### 特點
- **網路模式**：加密的 WireGuard 私有 Mesh 網狀網路 (`http://colab-model-station:8000`)
- **必要條件**：在 Colab Secrets 中設定 `TAILSCALE_AUTHKEY`。
- **適用場景**：生產級私有直連，可由本地終端機或 IDE 無縫發送推論請求，無任何公網暴露風險。

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. 以 Userspace 模式連線至 Tailscale Mesh 網路
bash engine.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. 啟動 FastAPI Diffusers 推論服務 (FLUX.1-schnell)
bash engine.sh diffusers start --model "black-forest-labs/FLUX.1-schnell" --precision fp8

# 3. 將 8000 埠發布至 Tailnet 內部
bash engine.sh tunnel tailscale serve 8000

# 4. 啟動閒置守護行程 (預設 30 分鐘自動釋放運算單元)
bash engine.sh watchdog start --port 8000 --timeout 1800

echo "================================================================"
echo "Diffusers Tailnet 連線端點: http://colab-model-station:8000"
echo "================================================================"


## 部署方案 B：Diffusers 服務 + Cloudflare 公網通道

### 特點
- **網路模式**：暫時性公網 HTTPS 網址 (`https://*.trycloudflare.com`)
- **必要條件**：零配置，不需任何帳號。
- **適用場景**：快速概念驗證、暫時對外展示或本地端未安裝 Tailscale 時使用。

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. 啟動 FastAPI Diffusers 服務
bash engine.sh diffusers start --model "black-forest-labs/FLUX.1-schnell" --precision fp8

# 2. 開啟 Cloudflare 公網穿透通道至 8000 埠
bash engine.sh tunnel cloudflare up 8000

# 3. 啟動閒置保護守護行程
bash engine.sh watchdog start --port 8000 --timeout 1800


## 步驟 2：筆記本內推論測試與動態 LoRA 切換

驗證推論服務是否正常回應，產出測試圖像並檢驗動態 LoRA 載入狀態。

In [ ]:
import requests
import base64
from IPython.display import display, Image

BASE_URL = "http://127.0.0.1:8000"

# 1. 健康檢查與顯存狀態檢驗
health = requests.get(f"{BASE_URL}/health").json()
print(f"服務狀態: {health['status']} | GPU: {health['gpu_name']} | 顯存佔用: {health['vram_allocated_mb']}MB")

# 2. 產生測試圖像
payload = {
    "prompt": "A futuristic scientific laboratory on Mars, hyperrealistic, cinematic lighting",
    "num_inference_steps": 4,
    "width": 1024,
    "height": 1024,
    "return_base64": True
}
res = requests.post(f"{BASE_URL}/v1/images/generations", json=payload).json()
print(f"生成完成，耗時: {res['generation_time_sec']}秒")

img_b64 = res['images'][0]['base64_data']
display(Image(base64.b64decode(img_b64)))


## 步驟 3：運算單元保護與執行階段釋放

> **重要提醒**：單純關閉瀏覽器分頁**不會**停止 Colab 虛擬機，計費運算單元將持續扣除至伺服器逾時為止。

工作完成後請務必執行此儲存格，立即釋放所有 GPU 顯存並中斷 Colab 執行階段。

In [ ]:
# ==============================================================================
# 步驟 3：終止 Colab 執行階段並停止計費
# ==============================================================================
from google.colab import runtime
print("正在中斷 Google Colab 執行階段以釋放運算單元...")
runtime.unassign()
